<a href="https://colab.research.google.com/github/weagan/In-Context-Learning/blob/main/lora_vs_icl_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRA Fine-tuning vs. In-Context Learning (ICL) — GPT-2

This notebook compares two adaptation strategies for a **generative model (GPT-2)**:

| Method | Description |
|--------|-------------|
| **Generative ICL** | Frozen GPT-2 + few-shot prompt; prediction parsed from generated text |
| **LoRA Fine-tuned GPT-2** | GPT-2 fine-tuned with LoRA adapters; same prompt format, same parsing logic |

Both methods use **identical prompt formats** and **identical output-parsing logic** so the comparison is fair.

**Task:** Predict which food a person likes from a short context sentence.

## 0. Install Dependencies

In [ ]:
# Install required libraries. Run once per environment.
!pip install -q transformers datasets peft accelerate matplotlib scikit-learn

## 1. Imports & Global Config

In [ ]:
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,  # needed to collate causal-LM batches correctly
    pipeline,
    set_seed,
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score

# ── Device ──────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ── Reproducibility seeds ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

## 2. Synthetic Dataset

We generate 300 short sentences of the form:

> *"Alice lives in Paris. People in Paris love tacos. What food does Alice like?"*

The label is the food mentioned in the sentence. This is a simple retrieval task —
the answer is always present in the context — so we can cleanly measure whether
each method successfully learns the output format.

In [ ]:
def generate_example():
    """Generate one (text, label) pair with a random person, city, and food."""
    names  = ["Alice", "Bob", "Charlie", "Diana"]
    cities = ["Paris", "Rome", "Berlin", "Madrid"]
    foods  = ["pizza", "sushi", "tacos", "pasta"]

    person = random.choice(names)
    city   = random.choice(cities)
    food   = random.choice(foods)

    context  = f"{person} lives in {city}. People in {city} love {food}."
    question = f"What food does {person} like?"

    return {"text": context + " " + question, "label": food}


# ── Build raw data list with string labels ────────────────────────────────────
# NOTE: We keep a separate raw_data list so we can later build the generative
# fine-tuning dataset without fighting the integer-label version.
raw_data = [generate_example() for _ in range(300)]

# ── Build label <-> id mappings ───────────────────────────────────────────────
# Sort labels so the mapping is deterministic across runs.
LABELS   = sorted(set(d["label"] for d in raw_data))
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(LABELS)

print("Labels:", label2id)
print(f"Total examples: {len(raw_data)}")

# ── Build integer-label version for downstream use ────────────────────────────
# We do NOT mutate raw_data here; instead we produce a separate list.
data = [{"text": d["text"], "label": label2id[d["label"]]} for d in raw_data]

# ── Train / test split indices ────────────────────────────────────────────────
# We carve out indices explicitly so ICL and LoRA evaluation use the same set.
N_TRAIN = 240   # 80 %
train_data = data[:N_TRAIN]
test_data  = data[N_TRAIN:]   # 60 examples

print(f"Train: {len(train_data)} | Test: {len(test_data)}")

## 3. Load Base GPT-2 (for ICL)

We use HuggingFace's `pipeline` for convenient text generation.
The base model is **frozen** — no weights are updated during ICL.

In [ ]:
# text-generation pipeline wraps the model + tokenizer and handles batching
generator = pipeline("text-generation", model="gpt2", device=0 if device == "cuda" else -1)

# Grab the tokenizer from the pipeline for later use in the LoRA section
gpt2_tokenizer = generator.tokenizer
# GPT-2 has no pad token by default; reuse EOS so padding works
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

print("GPT-2 loaded. Vocab size:", gpt2_tokenizer.vocab_size)

## 4. Prompt Format (shared by ICL and LoRA evaluation)

Both approaches use **exactly the same** `Context: ... Output: ...` format.
- ICL prepends a few labelled examples before the test question.
- The LoRA-tuned model sees just the test prompt (no examples needed after fine-tuning).

In [ ]:
def create_few_shot_prompt(few_shot_examples, test_example):
    """
    Build a few-shot prompt from `few_shot_examples` followed by the test query.

    Args:
        few_shot_examples: list of dicts with keys 'text' (str) and 'label' (int)
        test_example:      dict with key 'text' (str)

    Returns:
        str: the fully formatted prompt
    """
    prompt = ""
    for ex in few_shot_examples:
        label_text = id2label[ex["label"]]          # convert int → food name
        prompt += f"Context: {ex['text']} Output: {label_text}.\n"
    # Test line ends with 'Output:' — the model must complete it
    prompt += f"Context: {test_example['text']} Output:"
    return prompt


# ── Sanity check ──────────────────────────────────────────────────────────────
sample_prompt = create_few_shot_prompt(train_data[:2], test_data[0])
print(sample_prompt)

## 5. Output Parser (shared by ICL and LoRA evaluation)

Both methods produce free-form text. We parse the prediction by checking
whether any known food label appears in the generated continuation.

> **Note:** This heuristic is intentionally simple. In practice you would
> constrain generation (e.g., force-decode only valid tokens) for a fairer
> comparison. Both methods are subject to the **same** brittleness here.

In [ ]:
def parse_food_from_generated_text(generated_text):
    """
    Extract a food label from the generated text that follows 'Output:'.

    Strategy:
      1. Split on 'Output:' and take the last segment.
      2. Scan that segment for any known food word.

    Returns:
        int | None: label id if found, else None
    """
    try:
        # The model echoes the prompt, so the answer lives after the last 'Output:'
        continuation = generated_text.split("Output:")[-1].strip().lower()
    except IndexError:
        return None

    # Check each food label; return the first match
    for food_label in LABELS:
        if food_label in continuation:
            return label2id[food_label]

    # No food label found in the continuation
    return None

## 6. Generative ICL Evaluation (frozen GPT-2)

We feed a **2-shot prompt** into the base GPT-2 for each test example and
parse the generated food label.

In [ ]:
def predict_icl(few_shot_examples, test_example, gen_pipeline, max_new_tokens=10):
    """
    Generate a prediction for `test_example` using in-context learning.

    Args:
        few_shot_examples: k labelled examples shown before the test query
        test_example:      the example we want to classify
        gen_pipeline:      HuggingFace text-generation pipeline
        max_new_tokens:    how many tokens the model may append

    Returns:
        int | None: predicted label id
    """
    prompt = create_few_shot_prompt(few_shot_examples, test_example)

    # Generate text.
    # pad_token_id=eos_token_id silences a HuggingFace warning about open-ended
    # generation lacking a padding token.
    output = gen_pipeline(
        prompt,
        max_new_tokens=max_new_tokens,
        num_return_sequences=1,
        pad_token_id=gen_pipeline.tokenizer.eos_token_id,
        do_sample=False,          # greedy decoding for reproducibility
    )
    generated_text = output[0]["generated_text"]
    return parse_food_from_generated_text(generated_text)


# ── Run ICL evaluation ────────────────────────────────────────────────────────
K_SHOT            = 2                  # number of labelled examples in the prompt
N_ICL_TEST        = len(test_data)     # test-set size (generation is slow)

# Use the first K_SHOT training examples as the fixed few-shot demos
icl_few_shot_demos = train_data[:K_SHOT]
# Evaluate on the first N_ICL_TEST examples of the held-out test set
icl_test_subset   = test_data[:N_ICL_TEST]

icl_preds  = []
icl_labels = []

for i, ex in enumerate(icl_test_subset):
    if i % 5 == 0:
        print(f"ICL: processing {i+1}/{N_ICL_TEST}...")
    icl_preds.append(predict_icl(icl_few_shot_demos, ex, generator))
    icl_labels.append(ex["label"])

# ── Compute accuracy, ignoring examples where the parser returned None ────────
valid_mask   = [p is not None for p in icl_preds]
valid_preds  = [p for p, m in zip(icl_preds,  valid_mask) if m]
valid_labels = [l for l, m in zip(icl_labels, valid_mask) if m]

if valid_preds:
    icl_acc = accuracy_score(valid_labels, valid_preds)
    print(f"\nGenerative ICL ({K_SHOT}-shot) Accuracy: {icl_acc:.2f}  "
          f"(parsed {len(valid_preds)}/{N_ICL_TEST} predictions)")
else:
    print("\nICL: no valid predictions parsed. Setting accuracy = 0.")
    icl_acc = 0.0

## 7. LoRA Fine-tuning on GPT-2

### 7.1 Prepare the Dataset for Causal LM Fine-tuning

We format every training example as a **complete string** the model should
learn to produce:

```
Context: <text> Output: <label>.
```

For causal language modelling, `labels` = `input_ids` (the model predicts
the next token at every position, including the output label).

In [ ]:
def format_for_generative_ft(example):
    """
    Format a data example as a complete 'Context: ... Output: <label>.' string.

    The model is trained to predict every token in this string, including the
    label at the end. We use raw_data (string labels) to avoid double-converting.
    """
    label_text = id2label[example["label"]]
    return {"text": f"Context: {example['text']} Output: {label_text}."}


# Build formatted HuggingFace datasets from the same train/test split
gen_train_ds = Dataset.from_list([format_for_generative_ft(d) for d in train_data])
gen_test_ds  = Dataset.from_list([format_for_generative_ft(d) for d in test_data])


def tokenize_fn(batch):
    """
    Tokenize a batch of strings.
    max_length=128 is comfortably larger than our sentences (~30 tokens).
    """
    return gpt2_tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )


gen_train_ds = gen_train_ds.map(tokenize_fn, batched=True)
gen_test_ds  = gen_test_ds.map(tokenize_fn,  batched=True)

# For causal LM, labels == input_ids (shifted internally by the model)
gen_train_ds = gen_train_ds.map(lambda b: {"labels": b["input_ids"]}, batched=True)
gen_test_ds  = gen_test_ds.map(lambda b: {"labels": b["input_ids"]},  batched=True)

# FIX: include 'labels' in set_format so Trainer can access it
gen_train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
gen_test_ds.set_format( type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"Fine-tuning datasets ready — train: {len(gen_train_ds)}, test: {len(gen_test_ds)}")
print("Sample columns:", gen_train_ds.column_names)

### 7.2 Load GPT-2 and Apply LoRA Adapters

LoRA injects small rank-decomposition matrices into the attention and MLP
projections. Only those matrices are trained (~1% of parameters), keeping
the base model frozen.

In [ ]:
# Load a fresh GPT-2 for fine-tuning (separate from the frozen ICL model)
gpt2_base = AutoModelForCausalLM.from_pretrained("gpt2")

# ── LoRA configuration ────────────────────────────────────────────────────────
# target_modules: GPT-2 uses Conv1D layers named c_attn, c_proj, c_fc.
# PEFT automatically detects the fan_in_fan_out flag for Conv1D.
lora_config = LoraConfig(
    r=8,                                        # rank of the low-rank matrices
    lora_alpha=16,                              # scaling factor (effective lr multiplier)
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

gpt2_lora = get_peft_model(gpt2_base, lora_config)
gpt2_lora.to(device)
gpt2_lora.print_trainable_parameters()
print("LoRA adapters applied.")

### 7.3 Train with HuggingFace Trainer

> **Bug fixed:** Older versions of `transformers` accepted a `tokenizer=`
> argument in `Trainer.__init__`. Newer versions removed it and use
> `processing_class=` (or a `DataCollator`) instead. We pass a
> `DataCollatorForLanguageModeling` to handle padding correctly.

In [ ]:
# DataCollator shifts labels automatically and handles variable-length batches
# mlm=False → causal LM (next-token prediction), not masked LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=gpt2_tokenizer,
    mlm=False,
)

training_args = TrainingArguments(
    output_dir="./gpt2_lora_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy="epoch",        # evaluate after every epoch
    save_strategy="no",           # don't checkpoint (saves disk space)
    report_to="none",             # disable W&B / TensorBoard
    seed=SEED,
)

# FIX: pass data_collator instead of tokenizer= (removed in newer transformers)
lora_trainer = Trainer(
    model=gpt2_lora,
    args=training_args,
    train_dataset=gen_train_ds,
    eval_dataset=gen_test_ds,
    data_collator=data_collator,
)

print("Starting LoRA fine-tuning...")
lora_trainer.train()
print("Fine-tuning complete.")

## 8. LoRA Fine-tuned GPT-2 Evaluation

We use the **same prompt format and parser** as the ICL section.
The difference is that the LoRA model receives only the test prompt
(no few-shot examples in the prefix) — after fine-tuning it should
have internalised the output format.

In [ ]:
def predict_lora(test_example, fine_tuned_model, tokenizer, max_new_tokens=10):
    """
    Generate a prediction for `test_example` using the LoRA fine-tuned model.

    The prompt format mirrors what the model was trained on:
        'Context: ... Output:'
    The model must complete 'Output:' with the correct food label.

    Args:
        test_example:      dict with key 'text' (str) and 'label' (int)
        fine_tuned_model:  the LoRA-adapted GPT-2
        tokenizer:         GPT-2 tokenizer
        max_new_tokens:    budget for the completion

    Returns:
        int | None: predicted label id
    """
    # No few-shot examples — the fine-tuned model should know the format already
    prompt = f"Context: {test_example['text']} Output:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    fine_tuned_model.eval()
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,      # greedy — same decoding strategy as ICL
        )

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return parse_food_from_generated_text(generated_text)


# ── Run LoRA evaluation on the same test subset used for ICL ──────────────────
N_LORA_TEST = len(test_data)          # keep sizes equal for a fair comparison
lora_test_subset = test_data[:N_LORA_TEST]

lora_preds  = []
lora_labels = []

for i, ex in enumerate(lora_test_subset):
    if i % 5 == 0:
        print(f"LoRA eval: processing {i+1}/{N_LORA_TEST}...")
    lora_preds.append(predict_lora(ex, gpt2_lora, gpt2_tokenizer))
    lora_labels.append(ex["label"])

# ── Compute accuracy ──────────────────────────────────────────────────────────
valid_mask_l  = [p is not None for p in lora_preds]
valid_preds_l = [p for p, m in zip(lora_preds,  valid_mask_l) if m]
valid_labs_l  = [l for l, m in zip(lora_labels, valid_mask_l) if m]

if valid_preds_l:
    lora_acc = accuracy_score(valid_labs_l, valid_preds_l)
    print(f"\nLoRA Fine-tuned GPT-2 Accuracy: {lora_acc:.2f}  "
          f"(parsed {len(valid_preds_l)}/{N_LORA_TEST} predictions)")
else:
    print("\nLoRA eval: no valid predictions parsed. Setting accuracy = 0.")
    lora_acc = 0.0

## 9. Final Comparison Plot

Both methods use GPT-2, the same prompt format, and the same output parser.
The LoRA method additionally trains on the task data — so any accuracy gain
reflects what LoRA fine-tuning buys over pure prompting.

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("=" * 48)
print(f"  Generative ICL (GPT-2, {K_SHOT}-shot): {icl_acc:.2f}")
print(f"  LoRA Fine-tuned (GPT-2):             {lora_acc:.2f}")
print("=" * 48)

# ── Bar chart ─────────────────────────────────────────────────────────────────
method_labels = [
    f"Generative ICL\n(GPT-2, {K_SHOT}-shot)",
    "LoRA Fine-tuned\n(GPT-2)",
]
values = [icl_acc, lora_acc]
colors = ["lightcoral", "lightgreen"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(method_labels, values, color=colors, width=0.4, edgecolor="grey")

# Annotate bars with exact accuracy
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.02,
        f"{val:.2f}",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
    )

ax.set_title("Accuracy Comparison: Generative ICL vs. LoRA Fine-tuned (GPT-2)",
             fontsize=13, pad=12)
ax.set_ylabel("Accuracy", fontsize=11)
ax.set_ylim(0, 1.15)
ax.axhline(y=0.25, color="grey", linestyle="--", linewidth=0.8, label="Random baseline (4 classes)")
ax.legend(fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("lora_vs_icl_comparison.png", dpi=150)
plt.show()
print("Plot saved to lora_vs_icl_comparison.png")